# Tabular Data Processing with Prior Labs MCP

In this recipe, we scrape hotel listings from the web, structure them into a table using an LLM, and then use a Haystack `Agent` equipped with [Prior Labs' TabPFN](https://priorlabs.ai/) tools to predict and fill in missing attributes.

**Services used:**
- [Firecrawl](https://www.firecrawl.dev/) — web scraping
- [Anthropic Claude](https://www.anthropic.com/) — LLM for extraction and agent reasoning
- [Prior Labs MCP](https://priorlabs.ai/) — tabular ML predictions via MCP tools

## Install dependencies

In [1]:
%pip install -q haystack-ai mcp-haystack trafilatura firecrawl-haystack


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Set up API keys

You'll need three API keys:
- **Anthropic API key** — get one at [console.anthropic.com](https://console.anthropic.com/)
- **Firecrawl API key** — get one at [firecrawl.dev](https://www.firecrawl.dev/)
- **Prior Labs MCP token** — get one at [ux.priorlabs.ai](https://ux.priorlabs.ai/)

In [ ]:
import os
from getpass import getpass

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
if "FIRECRAWL_API_KEY" not in os.environ:
    os.environ["FIRECRAWL_API_KEY"] = getpass("Enter your Firecrawl API key: ")
if "PRIOR_LABS_MCP_TOKEN" not in os.environ:
    os.environ["PRIOR_LABS_MCP_TOKEN"] = getpass("Enter your Prior Labs MCP token: ")

## Step 1: Scrape and structure hotel listings

We build a pipeline that fetches hotel search results with `FirecrawlCrawler` and passes the raw text to Claude, which extracts the data into a structured Markdown table. Any missing fields are left blank - we'll fill those in with ML predictions in the next step.

In [4]:
from haystack_integrations.components.generators.anthropic import AnthropicChatGenerator
from haystack_integrations.components.fetchers.firecrawl import FirecrawlCrawler
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack import Pipeline


pipeline = Pipeline()
pipeline.add_component("fetcher", FirecrawlCrawler())
pipeline.add_component(
    "prompt_builder", 
    ChatPromptBuilder(
        template=[
            ChatMessage.from_system(
                "You are a hotel booking assistant. Extract the relevant information from the text and summarize it in a concise manner, as a Markdown table with the following columns: \n- Hotel name\n- Location\n- Distance from the target\n- Price per night\n- Room type\n- User rating\n- Location rating\n- Number of reviews\n- Number and types of beds\n- Amenities\n- Room size\n- Whether breakfast is included\n- Public transport availability\n\nIf any of the information is missing, leave the corresponding cell empty. Always return all the hotels from the input, even if some of the information is missing."
            ),
            ChatMessage.from_user(
                "{% for doc in documents %}{{ doc.content }}\n{% endfor %}"
            ),
        ],
        required_variables=["documents"],
    ),
)
pipeline.add_component(
    "summarizer", 
    AnthropicChatGenerator(
        model="claude-opus-4-6",
        generation_kwargs={
            "max_tokens": 10000,
        }
    ),
)

pipeline.connect("fetcher.documents", "prompt_builder.documents")
pipeline.connect("prompt_builder.prompt", "summarizer.messages")

🚅 Components
  - fetcher: FirecrawlCrawler
  - prompt_builder: ChatPromptBuilder
  - summarizer: AnthropicChatGenerator
🛤️ Connections
  - fetcher.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> summarizer.messages (list[ChatMessage])

In [5]:
output = pipeline.run({
    "fetcher": {
        "urls": [
            "https://www.booking.com/searchresults.html?ss=Berlin%2C+Berlin+Federal+State%2C+Germany&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaLYBiAEBmAEzuAEHyAEM2AED6AEB-AEBiAIBqAIBuAKQgrTOBsACAdICJDllMDgyNTA1LTdlYmMtNDJkNi04NzFmLWUyYTRkNWM1NjM3ZtgCAeACAQ&aid=304142&lang=en-us&sb=1&src_elem=sb&src=index&dest_id=-1746443&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=xu&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=0da2508885520c7d&ac_meta=GhAwZGEyNTA4ODg1NTIwYzdkIAAoATICeHU6BmJlcmxpbg%3D%3D&checkin=2026-05-05&checkout=2026-05-08&group_adults=1&no_rooms=1&group_children=0",
        ]
    }
})

In [6]:
from IPython.display import Markdown, display

display(Markdown(output["summarizer"]["replies"][-1].text))

| Hotel Name | Location | Distance from Downtown | Price per Night | Room Type | User Rating | Location Rating | Number of Reviews | Beds | Amenities | Room Size | Breakfast Included | Public Transport |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Global Living I TheGraf - East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 3.9 miles | $130 | One-Bedroom Apartment | 8.6 Excellent | | 176 | 3 beds (1 bunk bed, 1 sofa bed, 1 queen) | 1 bedroom, 1 living room, 1 bathroom, 1 kitchen | 431 ft² | No | |
| Hotel Berlin, Berlin (Radisson Individuals) | Mitte, Berlin | 1.3 miles | $188 | Cozy Small Room | 8.2 Very Good | | 19,683 | 1 queen bed | Sustainability certification | | No | Subway Access |
| Lux 11 Berlin-Mitte | Mitte, Berlin | 1.5 miles | $169 | Superior Room | 8.7 Excellent | 9.5 | 2,076 | 1 queen bed | | | No | Subway Access |
| SANA Berlin Hotel | Charlottenburg-Wilmersdorf, Berlin | 2.1 miles | $136 | Deluxe Double Room | 8.7 Excellent | | 5,269 | 1 queen bed | | | No | Subway Access |
| Numa Berlin Arc | Friedrichshain-Kreuzberg, Berlin | 1 mile | $145 | Double Room | 8.3 Very Good | | 5,286 | 1 queen bed | | | No | Subway Access |
| Premier Inn Berlin Airport | Treptow-Köpenick, Berlin | 10.8 miles | $74 | Standard Double Room | 8.5 Very Good | | 9,901 | 1 full bed | | | No | |
| Hotel Gat Point Charlie | Mitte, Berlin | 0.7 miles | $111 | Budget Double Room | 8.5 Very Good | 9.5 | 7,586 | 1 double or 2 twins | Sustainability certification | | No | Subway Access |
| Myer's Hotel Berlin | Pankow, Berlin | 2 miles | $232 | Comfort Double or Twin Room | 9.0 Wonderful | | 998 | 1 double or 2 twins | | | Yes | Subway Access |
| Schulz Hotel Berlin Wall at the East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 2.4 miles | $107 | Standard Single Room | 8.6 Excellent | 9.3 | 19,879 | 1 twin bed | Free cancellation, No prepayment needed | | No | |
| Telegraphenamt | Mitte, Berlin | 0.9 miles | $276 | Cozy Room | 8.8 Excellent | 9.7 | 725 | 1 king bed | Sustainability certification | | No | |
| Numa Berlin Weinmeister | Mitte, Berlin | 1.3 miles | $147 | Large Room - Accessible | 8.5 Very Good | 9.5 | 1,896 | 1 queen bed | | | No | Subway Access |
| Motel One Berlin-Alexanderplatz | Mitte, Berlin | 1.5 miles | $165 | Standard Double Room | 8.7 Excellent | 9.6 | 16,286 | 1 queen bed | Sustainability certification | | No | Subway Access |
| Living Hotel Berlin Mitte | Mitte, Berlin | 1.4 miles | $168 | Business Double Apartment | 8.6 Excellent | | 1,483 | 1 queen bed | Sustainability certification; 1 bedroom, 1 bathroom | 248 ft² | No | Subway Access |
| Hotel 38 | Mitte, Berlin | 0.9 miles | $166 | Double Room | 8.5 Very Good | 9.6 | 4,994 | 1 double or 2 twins | | | Yes | Subway Access |
| Living Hotel Großer Kurfürst | Mitte, Berlin | 1.4 miles | $171 | Hypoallergenic Double Room | 8.7 Excellent | 9.4 | 3,321 | 1 queen bed | Sustainability certification | | No | Subway Access |
| Locke at East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 2.6 miles | $114 | City Studio | 9.1 Wonderful | 9.3 | 3,119 | 1 queen bed | 1 bathroom | 237 ft² | No | |
| The Social Hub Berlin Alexanderplatz | Mitte, Berlin | 1.7 miles | $174 | Standard Queen Room | 8.6 Excellent | 9.3 | 9,302 | 1 queen bed | Free cancellation | | No | Subway Access |
| Hotel Nikolai Residence | Mitte, Berlin | 1.3 miles | $181 | Double Room | 9.0 Wonderful | 9.7 | 2,556 | 1 double or 2 twins | | | No | Subway Access |
| Steigenberger Hotel Am Kanzleramt | Mitte, Berlin | 0.6 miles | $240 | Superior Plus Room | 8.8 Excellent | 9.5 | 7,641 | 1 double or 2 twins | Sustainability certification; Free cancellation, No prepayment needed | | No | Subway Access |
| IntercityHotel Berlin Hauptbahnhof | Mitte, Berlin | 0.7 miles | $184 | Double Room - Disability Access | 8.0 Very Good | 9.4 | 10,493 | 1 twin bed | Sustainability certification; Free cancellation, No prepayment needed | | No | Subway Access |
| art'otel berlin mitte (Radisson Hotels) | Mitte, Berlin | 1.4 miles | $213 | Art Room courtyard view | 8.9 Excellent | 9.3 | 3,327 | 1 double or 2 twins | Sustainability certification | | No | Subway Access |
| Numa Berlin Torstrasse | Mitte, Berlin | 1.6 miles | $158 | Medium Room | 8.4 Very Good | 9.4 | 2,766 | 1 queen bed | | | No | Subway Access |
| URBAN LOFT Berlin | Mitte, Berlin | 1.2 miles | $179 | Deluxe Double Room | 8.8 Excellent | | 4,663 | 1 queen bed | | | No | |
| Hotel Augusta Am Kurfürstendamm | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $144 | Superior Single Room | 8.1 Very Good | 9.4 | 2,965 | 1 twin bed | | | No | Subway Access |
| aletto Hotel Potsdamer Platz | Friedrichshain-Kreuzberg, Berlin | 1.1 miles | $124 | Double or Twin Room | 8.8 Excellent | | 8,907 | 2 twin beds | Sustainability certification | | No | Subway Access |
| Michelberger Hotel | Friedrichshain-Kreuzberg, Berlin | 3.1 miles | $151 | Small Double Room | 8.4 Very Good | | 2,288 | 1 king bed | | | No | Subway Access |
| THE GATE GARDEN Hotel | Mitte, Berlin | 1.1 miles | $163 | Classic Double Room with City View | 8.8 Excellent | | 721 | 1 queen bed | | | No | Subway Access |
| Capri by Fraser Berlin | Mitte, Berlin | 1.1 miles | $259 | Executive Studio | 9.0 Wonderful | 9.4 | 3,377 | 1 king bed | Sustainability certification | | No | Subway Access |

## Step 2: Connect to Prior Labs via MCP

[Prior Labs](https://priorlabs.ai/) exposes [TabPFN](https://github.com/PriorLabs/TabPFN), a pretrained tabular foundation model, through an MCP server. We load the toolset, which gives the agent tools for uploading datasets and running fit/predict operations.

In [7]:
from haystack_integrations.tools.mcp import MCPToolset, StreamableHttpServerInfo
from haystack.utils import Secret

# Get the server info from environment variables and create the toolset
# based on the tools exposed by the MCP server.
server_info = StreamableHttpServerInfo(
    url="https://api.priorlabs.ai/mcp/server",
    token=Secret.from_env_var("PRIOR_LABS_MCP_TOKEN"),
)
toolset = MCPToolset(server_info=server_info, eager_connect=True)
toolset.tools

[Tool(name='upload_dataset', description='Default entrypoint for file-based data.\n\nUse this whenever the dataset is referenced as a file path, attachment, or URL.\nDo not ask users to paste entire CSV contents into the conversation.\n\nInitializes a direct upload to cloud storage. Returns a dataset_id and a\npre-signed upload_url valid for 60 minutes.\n\nAfter calling this tool:\n  1. HTTP PUT the file bytes directly to upload_url.\n  2. Do NOT read the file contents into the conversation.\n  3. Pass the returned dataset_id to fit_and_predict_from_dataset\n     or predict_from_dataset.\n\nCall this tool once per file — once for train.csv, once for test.csv.\nThe dataset_id has no implicit train/test meaning; label it yourself.\n\nFor ChatGPT/Claude web sessions without code execution:\n  - still prefer this tool for real datasets\n  - if upload cannot be executed in-session, do not paste full files inline;\n    use an upload-capable MCP client/session.\n\nDo NOT call this when the da

## Step 3: Add an agent to fill in missing values

We extend the pipeline with a Haystack `Agent` that receives the structured hotel table and uses Prior Labs tools to predict missing attributes (location ratings, room sizes, transport options). Predicted values are marked in **bold** in the final output.

In [8]:
from haystack.components.agents import Agent

llm = AnthropicChatGenerator(
    model="claude-opus-4-6",
    generation_kwargs={
        "max_tokens": 10000,
    }
)

pipeline.add_component(
    "final_prompt_builder",
    ChatPromptBuilder(
        # We need to rewrite the previous assistant replies to user
        # messages, as this particular provider expects the conversation
        # to end with a user message.
        template=[
            ChatMessage.from_user(
                "{% for reply in replies %}"
                "{{ reply.text }}\n"
                "{% endfor %}"
            ),
        ],
        required_variables=["replies"],
    ),
)
pipeline.add_component(
    "agent",
    Agent(
        chat_generator=llm,
        tools=toolset,
        system_prompt="You are a data scientist working with tabular data with a specialization in travel and hospitality. You have access to a set of tools that you should use to predict the missing attributes of hotels based on the information provided in the input table.\nBefore you use any of the tools, make sure to prepare the hotel data appropriately, like you would do for any tabular dataset, and interpret the results to create a human-friendly table with a structure similar to the input table.\nReturn only a Markdown-formatted table with the same columns as the input, but with the missing attributes filled in. Do not return any text other than the table. Mark filled attributes in bold. Do not return the tool outputs directly, or your reasoning, but only interpret them and return only the final table.",
    ),
)

pipeline.connect("summarizer.replies", "final_prompt_builder.replies")
pipeline.connect("final_prompt_builder.prompt", "agent.messages")

🚅 Components
  - fetcher: FirecrawlCrawler
  - prompt_builder: ChatPromptBuilder
  - summarizer: AnthropicChatGenerator
  - final_prompt_builder: ChatPromptBuilder
  - agent: Agent
🛤️ Connections
  - fetcher.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> summarizer.messages (list[ChatMessage])
  - summarizer.replies -> final_prompt_builder.replies (list[ChatMessage])
  - final_prompt_builder.prompt -> agent.messages (list[ChatMessage])

In [9]:
output = pipeline.run({
    "fetcher": {
        "urls": [
            "https://www.booking.com/searchresults.html?ss=Berlin%2C+Berlin+Federal+State%2C+Germany&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaLYBiAEBmAEzuAEHyAEM2AED6AEB-AEBiAIBqAIBuAKQgrTOBsACAdICJDllMDgyNTA1LTdlYmMtNDJkNi04NzFmLWUyYTRkNWM1NjM3ZtgCAeACAQ&aid=304142&lang=en-us&sb=1&src_elem=sb&src=index&dest_id=-1746443&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=xu&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=0da2508885520c7d&ac_meta=GhAwZGEyNTA4ODg1NTIwYzdkIAAoATICeHU6BmJlcmxpbg%3D%3D&checkin=2026-05-05&checkout=2026-05-08&group_adults=1&no_rooms=1&group_children=0",
        ]
    }
})

In [10]:
print(output["agent"]["last_message"].text)

Now let me think about the results and reason about them carefully.

**Location Rating predictions** (rounded to 1 decimal):
- Global Living I TheGraf: 9.4 → 9.4
- Hotel Berlin, Berlin: 9.5 → 9.5
- SANA Berlin Hotel: 9.4 → 9.4 (but let me reconsider — it's in Charlottenburg, farther from downtown)
- Numa Berlin Arc: 9.5 → 9.5 (Friedrichshain, 1 mile, close)
- Premier Inn Berlin Airport: 9.4 → but this is 10.8 miles from downtown at the airport, so a lower location rating makes more sense. Let me reconsider... The model predicted 9.4, but given it's an airport hotel far from downtown, a lower rating seems more realistic. However, the training data doesn't have any examples that far out. The model only has data points relatively close to downtown. I'll round to a sensible value. Actually, looking at the training data, all location ratings range from 9.3 to 9.7 — these are Berlin hotels that all tend to get high location ratings. The airport hotel at 10.8 miles would likely have a notably

## Conclusion

LLMs are remarkably good at turning messy, unstructured web content into clean, structured tables, but they struggle with numerical reasoning and will often leave gaps where data wasn't explicitly present on the page. This is where classical tabular ML still shines: given a few known examples, TabPFN can extrapolate missing values with no fine-tuning required.

The combination is powerful: use an LLM to handle the unstructured-to-structured step, then hand off to a tabular model for the gaps. Haystack's `Agent` with MCP tools makes it straightforward to wire these two worlds together in a single pipeline.